# **Step 1: Getting Datasets**

# **Step 1.1: Getting file from Kaggle(1st Source of Dataset)**

In [ ]:
import pandas as pd
import time
import numpy as np
import requests

# link to dataser: https://www.kaggle.com/datasets/mexwell/ucs-satellite-database?select=satellites.csv

# CSV file normally contains ',' as separator but this file contains';' as column separator so below code is done accordingly
df_satellites = pd.read_csv("Input_Files/satellites.csv", sep=";", engine="python")
df_satellites.head()


,"Name of Satellite, Alternate Names",Current Official Name of Satellite,Country/Org of UN Registry,Country of Operator/Owner,Operator/Owner,Users,Purpose,Detailed Purpose,Class of Orbit,Type of Orbit,...,Unnamed: 58,Unnamed: 59,Unnamed: 60,Unnamed: 61,Unnamed: 62,Unnamed: 63,Unnamed: 64,Unnamed: 65,Unnamed: 66,Unnamed: 67
0,1HOPSAT-TD (1st-generation High Optical Perfor...,1HOPSAT-TD,NR,USA,Hera Systems,Commercial,Earth Observation,Infrared Imaging,LEO,Non-Polar Inclined,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AAC AIS-Sat1 (Kelpie 1),AAC AIS-Sat1 (Kelpie 1),United Kingdom,United Kingdom,AAC Clyde Space,Commercial,Earth Observation,Automatic Identification System (AIS),LEO,Sun-Synchronous,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Aalto-1,Aalto-1,Finland,Finland,Aalto University,Civil,Technology Development,NaN,LEO,Sun-Synchronous,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AAt-4,AAt-4,Denmark,Denmark,University of Aalborg,Civil,Earth Observation,Automatic Identification System (AIS),LEO,Sun-Synchronous,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"ABS-2 (Koreasat-8, ST-3)",ABS-2,NR,Multinational,Asia Broadcast Satellite Ltd.,Commercial,Communications,NaN,GEO,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# **Step 1.2: Getting file from Celestrak API (2nd Source of Dataset)**

In [ ]:
is_owner = 'Y'
is_owner = input("Are you owner of this file(Y/N):(if yes you need to fetach API(refreshing each 2hours), \nwe will provide you the datafile directly)")

if is_owner == 'Y':
    url = "https://celestrak.org/NORAD/elements/gp.php?GROUP=starlink&FORMAT=json"

    headers = {
        "User-Agent": "my-satellite-script/1.0 (contact: supriyagosavi14@gmail.com)",
        "Accept": "application/json"
    }

    response = requests.get(url, headers=headers)

    print(response.status_code)
    print(response.text[:200])  # inspect response safely before .json()

    data = response.json()
    df_live = pd.DataFrame(data)

    df_live.to_csv("Input_Files/df_live.csv", index=False)
else:
    df_live = pd.read_csv("Input_Files/df_live.csv")

200
[{"OBJECT_NAME":"STARLINK-1008","OBJECT_ID":"2019-074B","EPOCH":"2026-04-16T03:10:57.891072","MEAN_MOTION":15.34844011,"ECCENTRICITY":0.00031711,"INCLINATION":53.1552,"RA_OF_ASC_NODE":4.1211,"ARG_OF_P


# **Step 1.3: Getting file from Spacestrak API (3nd Source of Dataset)**

In [ ]:
is_owner = 'Y'
is_owner = input("Are you owner of this file(Y/N):(if yes you need to provide authorization details for API and if you not type No, \nwe will provide you the datafile directly)")

if is_owner == 'Y':
    USERNAME = input("Enter the username: ")
    PASSWORD = input("Enter the password(Hint:SAS): ")

    login_url = "https://www.space-track.org/ajaxauth/login"

    query_url = (
        "https://www.space-track.org/basicspacedata/query/"
        "class/satcat/orderby/INTLDES%20asc/format/json"
    )

    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept": "application/json,text/plain,*/*",
        "Content-Type": "application/x-www-form-urlencoded"
    }

    with requests.Session() as session:
        # LOGIN
        login_data = {
            "identity": USERNAME,
            "password": PASSWORD
        }

        resp = session.post(login_url, data=login_data, headers=headers)

        print("Login status:", resp.status_code)
        print("Login response preview:", resp.text[:200])

        # TEST AUTH
        if "Invalid" in resp.text or resp.status_code != 200:
            raise Exception("Login failed — check credentials or account access")

        # DATA REQUEST
        response = session.get(query_url, headers=headers)

        print("Query status:", response.status_code)
        print(response.text[:1000])

        data = response.json()
        space_track_data = pd.DataFrame(data)

        space_track_data.to_csv("Input_Files/space_track_data.csv", index=False)
else:
    space_track_data = pd.read_csv("Input_Files/space_track_data.csv")


Login status: 200
Login response preview: ""
Query status: 200
[{"INTLDES":"1957-001A","NORAD_CAT_ID":"1","OBJECT_TYPE":"ROCKET BODY","SATNAME":"SL-1 R\/B","COUNTRY":"CIS","LAUNCH":"1957-10-04","SITE":"TTMTR","DECAY":"1957-12-01","PERIOD":"96.19","INCLINATION":"65.10","APOGEE":"938","PERIGEE":"214","COMMENT":null,"COMMENTCODE":"4","RCSVALUE":"0","RCS_SIZE":"LARGE","FILE":"1","LAUNCH_YEAR":"1957","LAUNCH_NUM":"1","LAUNCH_PIECE":"A","CURRENT":"Y","OBJECT_NAME":"SL-1 R\/B","OBJECT_ID":"1957-001A","OBJECT_NUMBER":"1"},{"INTLDES":"1957-001B","NORAD_CAT_ID":"2","OBJECT_TYPE":"PAYLOAD","SATNAME":"SPUTNIK 1","COUNTRY":"CIS","LAUNCH":"1957-10-04","SITE":"TTMTR","DECAY":"1958-01-03","PERIOD":"96.10","INCLINATION":"65.00","APOGEE":"1080","PERIGEE":"64","COMMENT":null,"COMMENTCODE":null,"RCSVALUE":"0","RCS_SIZE":null,"FILE":"7179","LAUNCH_YEAR":"1957","LAUNCH_NUM":"1","LAUNCH_PIECE":"B","CURRENT":"Y","OBJECT_NAME":"SPUTNIK 1","OBJECT_ID":"1957-001B","OBJECT_NUMBER":"2"},{"INTLDES":"1957-002A","NOR

# **Step 2 : Cleaning Datasets**

# **Step 2.1: Cleaning of Dataset from Kaggle: Satellite.csv**

In [8]:
df_satellites.head(3)

,"Name of Satellite, Alternate Names",Current Official Name of Satellite,Country/Org of UN Registry,Country of Operator/Owner,Operator/Owner,Users,Purpose,Detailed Purpose,Class of Orbit,Type of Orbit,...,Unnamed: 58,Unnamed: 59,Unnamed: 60,Unnamed: 61,Unnamed: 62,Unnamed: 63,Unnamed: 64,Unnamed: 65,Unnamed: 66,Unnamed: 67
0,1HOPSAT-TD (1st-generation High Optical Perfor...,1HOPSAT-TD,NR,USA,Hera Systems,Commercial,Earth Observation,Infrared Imaging,LEO,Non-Polar Inclined,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AAC AIS-Sat1 (Kelpie 1),AAC AIS-Sat1 (Kelpie 1),United Kingdom,United Kingdom,AAC Clyde Space,Commercial,Earth Observation,Automatic Identification System (AIS),LEO,Sun-Synchronous,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Aalto-1,Aalto-1,Finland,Finland,Aalto University,Civil,Technology Development,NaN,LEO,Sun-Synchronous,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
df_satellites.columns

Index(['Name of Satellite, Alternate Names',
       'Current Official Name of Satellite', 'Country/Org of UN Registry',
       'Country of Operator/Owner', 'Operator/Owner', 'Users', 'Purpose',
       'Detailed Purpose', 'Class of Orbit', 'Type of Orbit',
       'Longitude of GEO (degrees)', 'Perigee (km)', 'Apogee (km)',
       'Eccentricity', 'Inclination (degrees)', 'Period (minutes)',
       'Launch Mass (kg.)', 'Dry Mass (kg.)', 'Power (watts)',
       'Date of Launch', 'Expected Lifetime (yrs.)', 'Contractor',
       'Country of Contractor', 'Launch Site', 'Launch Vehicle',
       'COSPAR Number', 'NORAD Number', 'Comments', 'Unnamed: 28',
       'Source Used for Orbital Data', 'Source', 'Source.1', 'Source.2',
       'Source.3', 'Source.4', 'Source.5', 'Source.6', 'Unnamed: 37',
       'Unnamed: 38', 'Unnamed: 39', 'Unnamed: 40', 'Unnamed: 41',
       'Unnamed: 42', 'Unnamed: 43', 'Unnamed: 44', 'Unnamed: 45',
       'Unnamed: 46', 'Unnamed: 47', 'Unnamed: 48', 'Unnamed: 49',
  

Removing unnecessary added columns

In [10]:
df_satellites = df_satellites.loc[:, ~df_satellites.columns.str.contains('^Unnamed')]
df_satellites.columns

Index(['Name of Satellite, Alternate Names',
       'Current Official Name of Satellite', 'Country/Org of UN Registry',
       'Country of Operator/Owner', 'Operator/Owner', 'Users', 'Purpose',
       'Detailed Purpose', 'Class of Orbit', 'Type of Orbit',
       'Longitude of GEO (degrees)', 'Perigee (km)', 'Apogee (km)',
       'Eccentricity', 'Inclination (degrees)', 'Period (minutes)',
       'Launch Mass (kg.)', 'Dry Mass (kg.)', 'Power (watts)',
       'Date of Launch', 'Expected Lifetime (yrs.)', 'Contractor',
       'Country of Contractor', 'Launch Site', 'Launch Vehicle',
       'COSPAR Number', 'NORAD Number', 'Comments',
       'Source Used for Orbital Data', 'Source', 'Source.1', 'Source.2',
       'Source.3', 'Source.4', 'Source.5', 'Source.6'],
      dtype='object')

In [11]:
df_satellites.head()

,"Name of Satellite, Alternate Names",Current Official Name of Satellite,Country/Org of UN Registry,Country of Operator/Owner,Operator/Owner,Users,Purpose,Detailed Purpose,Class of Orbit,Type of Orbit,...,NORAD Number,Comments,Source Used for Orbital Data,Source,Source.1,Source.2,Source.3,Source.4,Source.5,Source.6
0,1HOPSAT-TD (1st-generation High Optical Perfor...,1HOPSAT-TD,NR,USA,Hera Systems,Commercial,Earth Observation,Infrared Imaging,LEO,Non-Polar Inclined,...,44859,Pathfinder for planned earth observation const...,JMSatcat/3_20,https://spaceflightnow.com/2019/12/11/indias-5...,https://www.herasys.com/,NaN,NaN,NaN,NaN,NaN
1,AAC AIS-Sat1 (Kelpie 1),AAC AIS-Sat1 (Kelpie 1),United Kingdom,United Kingdom,AAC Clyde Space,Commercial,Earth Observation,Automatic Identification System (AIS),LEO,Sun-Synchronous,...,55107,Provide AIS information to Orbcomm.,JMSatcat/9_23,https://www.aac-clyde.space/articles/aac-clyde...,NaN,NaN,NaN,NaN,NaN,NaN
2,Aalto-1,Aalto-1,Finland,Finland,Aalto University,Civil,Technology Development,NaN,LEO,Sun-Synchronous,...,42775,Technology development and education.,JMSatcat/10_17,https://directory.eoportal.org/web/eoportal/sa...,NaN,http://www.planet4589.org/space/log/satcat.txt,NaN,NaN,NaN,NaN
3,AAt-4,AAt-4,Denmark,Denmark,University of Aalborg,Civil,Earth Observation,Automatic Identification System (AIS),LEO,Sun-Synchronous,...,41460,Carries AIS system.,Space50,http://spaceflightnow.com/2016/04/26/soyuz-bla...,NaN,http://space50.org/objekt.php?mot=2016-025E&ja...,NaN,NaN,NaN,NaN
4,"ABS-2 (Koreasat-8, ST-3)",ABS-2,NR,Multinational,Asia Broadcast Satellite Ltd.,Commercial,Communications,NaN,GEO,NaN,...,39508,"32 C-band, 51 Ku-band, and 6 Ka-band transpond...",ZARYA,http://www.absatellite.net/satellite-fleet/?sa...,NaN,http://www.zarya.info/Diaries/Launches/Launche...,http://www.absatellite.net/2010/10/13/asia-bro...,http://www.spacenews.com/article/satellite-tel...,NaN,NaN


Keep everything up to and including NORAD Number:

In [12]:
df_satellites = df_satellites.loc[:, :'NORAD Number']
df_satellites.columns

Index(['Name of Satellite, Alternate Names',
       'Current Official Name of Satellite', 'Country/Org of UN Registry',
       'Country of Operator/Owner', 'Operator/Owner', 'Users', 'Purpose',
       'Detailed Purpose', 'Class of Orbit', 'Type of Orbit',
       'Longitude of GEO (degrees)', 'Perigee (km)', 'Apogee (km)',
       'Eccentricity', 'Inclination (degrees)', 'Period (minutes)',
       'Launch Mass (kg.)', 'Dry Mass (kg.)', 'Power (watts)',
       'Date of Launch', 'Expected Lifetime (yrs.)', 'Contractor',
       'Country of Contractor', 'Launch Site', 'Launch Vehicle',
       'COSPAR Number', 'NORAD Number'],
      dtype='object')

Standardise the Column names: All in lower case and without spaces

In [13]:
df_satellites.columns = df_satellites.columns.str.lower().str.replace(' ', '_')
df_satellites.columns


Index(['name_of_satellite,_alternate_names',
       'current_official_name_of_satellite', 'country/org_of_un_registry',
       'country_of_operator/owner', 'operator/owner', 'users', 'purpose',
       'detailed_purpose', 'class_of_orbit', 'type_of_orbit',
       'longitude_of_geo_(degrees)', 'perigee_(km)', 'apogee_(km)',
       'eccentricity', 'inclination_(degrees)', 'period_(minutes)',
       'launch_mass_(kg.)', 'dry_mass_(kg.)', 'power_(watts)',
       'date_of_launch', 'expected_lifetime_(yrs.)', 'contractor',
       'country_of_contractor', 'launch_site', 'launch_vehicle',
       'cospar_number', 'norad_number'],
      dtype='object')

In [14]:
df_satellites.columns = (
    df_satellites.columns
    .str.lower()
    .str.replace('(', '_', regex=False)   # replace ( with _
    .str.replace('/', '_', regex=False)   # replace ( with _
    .str.replace(')', '', regex=False)    # remove )
    .str.replace(r'[^\w\s]', '', regex=True)  # remove other special chars
    .str.replace(' ', '_')                # spaces → underscore
    .str.replace('__', '_')
)
df_satellites.columns

Index(['name_of_satellite_alternate_names',
       'current_official_name_of_satellite', 'country_org_of_un_registry',
       'country_of_operator_owner', 'operator_owner', 'users', 'purpose',
       'detailed_purpose', 'class_of_orbit', 'type_of_orbit',
       'longitude_of_geo_degrees', 'perigee_km', 'apogee_km', 'eccentricity',
       'inclination_degrees', 'period_minutes', 'launch_mass_kg',
       'dry_mass_kg', 'power_watts', 'date_of_launch', 'expected_lifetime_yrs',
       'contractor', 'country_of_contractor', 'launch_site', 'launch_vehicle',
       'cospar_number', 'norad_number'],
      dtype='object')

Decide Columns to keep for analysis

In [15]:
cols_to_keep = [
    'current_official_name_of_satellite',
    'country_of_operator_owner',
    'operator_owner',
    'users',
    'purpose',
    'class_of_orbit',
    'type_of_orbit',
    'perigee_km',
    'apogee_km',
    'perigee_km',
    'launch_mass_kg',
    'date_of_launch',
    'expected_lifetime_yrs',
    'launch_vehicle',
    'norad_number'
]

new_satellite_data = df_satellites[cols_to_keep]
new_satellite_data.head(3)

,current_official_name_of_satellite,country_of_operator_owner,operator_owner,users,purpose,class_of_orbit,type_of_orbit,perigee_km,apogee_km,perigee_km,launch_mass_kg,date_of_launch,expected_lifetime_yrs,launch_vehicle,norad_number
0,1HOPSAT-TD,USA,Hera Systems,Commercial,Earth Observation,LEO,Non-Polar Inclined,566.0,576.0,566.0,22.0,12/11/19,"0,5",PSLV,44859
1,AAC AIS-Sat1 (Kelpie 1),United Kingdom,AAC Clyde Space,Commercial,Earth Observation,LEO,Sun-Synchronous,637.0,654.0,637.0,4.0,1/3/23,NaN,Falcon 9,55107
2,Aalto-1,Finland,Aalto University,Civil,Technology Development,LEO,Sun-Synchronous,497.0,517.0,497.0,5.0,6/23/17,2,PSLV,42775


Check for Duplicates

In [16]:
new_satellite_data.duplicated().sum()

np.int64(7)

Dropping Duplicates

In [17]:
new_satellite_data=new_satellite_data.drop_duplicates()
new_satellite_data.duplicated().sum()

np.int64(0)

Checking and correcting datatypes

In [18]:
new_satellite_data.dtypes

current_official_name_of_satellite     object
country_of_operator_owner              object
operator_owner                         object
users                                  object
purpose                                object
class_of_orbit                         object
type_of_orbit                          object
perigee_km                            float64
apogee_km                             float64
perigee_km                            float64
launch_mass_kg                        float64
date_of_launch                         object
expected_lifetime_yrs                  object
launch_vehicle                         object
norad_number                            int64
dtype: object

In [21]:
# Convert strings with potential non-numeric characters to numbers
new_satellite_data['launch_mass_kg'] = pd.to_numeric(new_satellite_data['launch_mass_kg'], errors='coerce')
new_satellite_data['date_of_launch'] = pd.to_datetime(new_satellite_data['date_of_launch'], errors='coerce')
# Convert to float, turning non-numeric strings into NaN (Not a Number)
new_satellite_data['expected_lifetime_yrs'] = pd.to_numeric(new_satellite_data['expected_lifetime_yrs'], errors='coerce')

In [20]:
new_satellite_data.dtypes

current_official_name_of_satellite            object
country_of_operator_owner                     object
operator_owner                                object
users                                         object
purpose                                       object
class_of_orbit                                object
type_of_orbit                                 object
perigee_km                                   float64
apogee_km                                    float64
perigee_km                                   float64
launch_mass_kg                               float64
date_of_launch                        datetime64[ns]
expected_lifetime_yrs                        float64
launch_vehicle                                object
norad_number                                   int64
dtype: object

Checking for Null values

In [22]:
round((new_satellite_data.isna().sum() / len(new_satellite_data)) * 100)

current_official_name_of_satellite     0.0
country_of_operator_owner              0.0
operator_owner                         0.0
users                                  0.0
purpose                                0.0
class_of_orbit                         0.0
type_of_orbit                          9.0
perigee_km                             0.0
apogee_km                              0.0
perigee_km                             0.0
launch_mass_kg                         3.0
date_of_launch                         0.0
expected_lifetime_yrs                 28.0
launch_vehicle                         0.0
norad_number                           0.0
dtype: float64

Handle missing values in column Expected_lifetime_yrs

In [23]:
#  Fill Lifetime by the median of its Orbit Class
new_satellite_data['expected_lifetime_yrs'] = new_satellite_data['expected_lifetime_yrs'].fillna(
    new_satellite_data.groupby('class_of_orbit')['expected_lifetime_yrs'].transform('median')
)

#  Fill any remaining tiny gaps with a global constant
new_satellite_data.fillna({'expected_lifetime_yrs': 5}, inplace=True)

# Fill missing Orbit Type with the most common type (mode) within that Orbit Class
new_satellite_data['type_of_orbit'] = new_satellite_data.groupby('class_of_orbit')['type_of_orbit'].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else "Unknown")
)

# handling Using Mean (Average)
new_satellite_data['launch_mass_kg'] = new_satellite_data['launch_mass_kg'].fillna(
    new_satellite_data['launch_mass_kg'].mean()
)

In [24]:
round((new_satellite_data.isna().sum() / len(new_satellite_data)) * 100)

current_official_name_of_satellite    0.0
country_of_operator_owner             0.0
operator_owner                        0.0
users                                 0.0
purpose                               0.0
class_of_orbit                        0.0
type_of_orbit                         0.0
perigee_km                            0.0
apogee_km                             0.0
perigee_km                            0.0
launch_mass_kg                        0.0
date_of_launch                        0.0
expected_lifetime_yrs                 0.0
launch_vehicle                        0.0
norad_number                          0.0
dtype: float64

In [25]:
new_satellite_data.describe()

,perigee_km,apogee_km,perigee_km,launch_mass_kg,date_of_launch,expected_lifetime_yrs,norad_number
count,7546.000000,7546.000000,7546.000000,7553.000000,7550,7553.000000,7553.000000
mean,433.132687,442.220914,433.132687,199.874653,2019-12-04 01:38:59.284768,5.145505,47904.287568
min,1.000000,1.000000,1.000000,1.000000,1988-09-29 00:00:00,1.000000,7530.000000
25%,348.000000,351.000000,348.000000,43.000000,2019-12-26 00:00:00,4.000000,44903.000000
50%,538.000000,541.000000,538.000000,260.000000,2021-05-09 00:00:00,4.000000,48445.000000
75%,549.000000,561.000000,549.000000,260.000000,2022-07-11 00:00:00,4.000000,53083.000000
max,989.000000,997.000000,989.000000,980.000000,2074-11-15 00:00:00,30.000000,62684.000000
std,215.240469,220.738180,215.240469,161.575586,NaN,3.183667,6557.797209


Seeing values of Categorical Columns to see if any outliers or some other issues

Cleaning Users Column

In [26]:
new_satellite_data['users'].unique()

array(['Commercial', 'Civil', 'Government', 'Military',
       'Military/Commercial', 'Government/Military',
       'Military/Government', 'Government/Civil', 'Military/Civil',
       'Commercial/Civil', 'Civil/Commercial', 'Government/Commercial',
       'Commercial/Government', 'Government/Commercial/Military',
       'Civil/Government', 'Civil/Military', 'Commercial ',
       'Commercial/Military', 'Government ', 'Military '], dtype=object)

In [27]:
# clean whitespace
new_satellite_data['users'] = new_satellite_data['users'].str.strip()

In [28]:
# normalize order of multi-label categories
new_satellite_data['users'] = (
    new_satellite_data['users']
    .str.split('/')
    .apply(lambda x: '/'.join(sorted([i.strip() for i in x])))
)
new_satellite_data['users'].unique()

array(['Commercial', 'Civil', 'Government', 'Military',
       'Commercial/Military', 'Government/Military', 'Civil/Government',
       'Civil/Military', 'Civil/Commercial', 'Commercial/Government',
       'Commercial/Government/Military'], dtype=object)

Cleaning Column Purpose

In [29]:
new_satellite_data['purpose'].unique()

array(['Earth Observation', 'Technology Development', 'Communications',
       'Earth Science', 'Space Science',
       'Space Science/Technology Demonstration',
       'Communications/Technology Development',
       'Communications/Maritime Tracking', 'Technology Demonstration',
       'Unknown', 'Navigation/Global Positioning',
       'Earth Observation/Technology Development', 'Earth Observation ',
       'Earth Observation/Communications', 'Earth/Space Observation',
       'Educational', 'Earth Observation/Earth Science', 'Platform',
       'Earth Observation/Space Science', 'Earth Observation/Navigation',
       'Communications/Navigation', 'Space Observation', 'Surveillance',
       'Navigation/Regional Positioning',
       'Space Science/Technology Development',
       'Mission Extension Technology', 'Earth Science/Earth Observation',
       'Earth Observation/Communications/Space Science', 'Meteorological',
       'Technology Development/Educational', 'Satellite Positioning'],


In [30]:
# strip whitespaces
new_satellite_data['purpose'] = new_satellite_data['purpose'].str.strip()

In [31]:
# normalize ordering
new_satellite_data['purpose'] = new_satellite_data['purpose'].apply(
    lambda x: '/'.join(sorted([i.strip() for i in x.split('/')]))
)
new_satellite_data['purpose'].unique()

array(['Earth Observation', 'Technology Development', 'Communications',
       'Earth Science', 'Space Science',
       'Space Science/Technology Demonstration',
       'Communications/Technology Development',
       'Communications/Maritime Tracking', 'Technology Demonstration',
       'Unknown', 'Global Positioning/Navigation',
       'Earth Observation/Technology Development',
       'Communications/Earth Observation', 'Earth/Space Observation',
       'Educational', 'Earth Observation/Earth Science', 'Platform',
       'Earth Observation/Space Science', 'Earth Observation/Navigation',
       'Communications/Navigation', 'Space Observation', 'Surveillance',
       'Navigation/Regional Positioning',
       'Space Science/Technology Development',
       'Mission Extension Technology',
       'Communications/Earth Observation/Space Science', 'Meteorological',
       'Educational/Technology Development', 'Satellite Positioning'],
      dtype=object)

In [32]:
# removing duplicate info like
# Global Positioning/Navigation and Navigation/Global Positioning
new_satellite_data['purpose'].value_counts().head(15)

purpose
Communications                              5507
Earth Observation                           1238
Technology Development                       372
Global Positioning/Navigation                142
Space Science                                 99
Technology Demonstration                      64
Earth Science                                 28
Surveillance                                  20
Navigation/Regional Positioning               13
Unknown                                       10
Space Observation                              9
Earth Observation/Navigation                   9
Earth Observation/Technology Development       7
Meteorological                                 6
Communications/Maritime Tracking               5
Name: count, dtype: int64

Cleaning column country_of_operator_owner

In [33]:
new_satellite_data['country_of_operator_owner'].unique()

array(['USA', 'United Kingdom', 'Finland', 'Denmark', 'Multinational',
       'Israel', 'Austria', 'ESA', 'Lithuania', 'Norway', 'Spain',
       'United Arab Emirates', 'Kazakhstan', 'Algeria', 'Japan', 'Brazil',
       'Russia', 'South Korea', 'Angola', 'Canada', 'Argentina',
       'USA/Argentina', 'China', 'Belgium', 'Turkey', 'Luxembourg',
       'Switzerland', 'India', 'France/Italy', 'Singapore', 'Azerbaijan',
       'Bangladesh', 'Czech Republic', 'Germany', 'China ', 'Belarus',
       'Netherlands', 'Indonesia', 'France', 'Australia', 'Bulgaria',
       'China/Brazil', 'China/France', 'Tunisia', 'Taiwan', 'Italy',
       'Mexico', 'Ecuador', 'Egypt', 'USA/Canada/Japan',
       'USA/Japan/Brazil', 'Ethiopia', 'Colombia', 'USA/Japan',
       'USA/Germany', 'France/Italy/Belgium/Spain/Greece', 'Greece',
       'Greece/United Kingdom', 'United Kingdom/ESA', 'Malaysia',
       'USA/India/Singapore/Taiwan', 'ESA/Russia', 'Thailand',
       'USA/France', 'Japan/Singapore', 'Jordan', '

In [34]:
# "China " with trailing spaces, so it’s better to normalize first:
new_satellite_data['country_of_operator_owner'] = new_satellite_data['country_of_operator_owner'].str.strip()

# apply the grouping
new_satellite_data['country_of_operator_owner'] = new_satellite_data['country_of_operator_owner'].str.strip().apply(
    lambda x: 'Multinational_countries' if '/' in str(x) else x
)


In [35]:
new_satellite_data['country_of_operator_owner'].unique()

array(['USA', 'United Kingdom', 'Finland', 'Denmark', 'Multinational',
       'Israel', 'Austria', 'ESA', 'Lithuania', 'Norway', 'Spain',
       'United Arab Emirates', 'Kazakhstan', 'Algeria', 'Japan', 'Brazil',
       'Russia', 'South Korea', 'Angola', 'Canada', 'Argentina',
       'Multinational_countries', 'China', 'Belgium', 'Turkey',
       'Luxembourg', 'Switzerland', 'India', 'Singapore', 'Azerbaijan',
       'Bangladesh', 'Czech Republic', 'Germany', 'Belarus',
       'Netherlands', 'Indonesia', 'France', 'Australia', 'Bulgaria',
       'Tunisia', 'Taiwan', 'Italy', 'Mexico', 'Ecuador', 'Egypt',
       'Ethiopia', 'Colombia', 'Greece', 'Malaysia', 'Thailand', 'Jordan',
       'Iran', 'Saudi Arabia', 'Kuwait', 'Laos', 'South Africa',
       'Vietnam', 'Morocco', 'Slovenia', 'Sinapore', 'Nigeria', 'Sweden',
       'Pakistan', 'Peru', 'New Zealand', 'Chile', 'Ukraine', 'Hungary',
       'Monaco', 'Nepal', 'Sudan', 'Poland', 'Kenya', 'Iraq', 'Bolivia',
       'Estonia', 'Venezuela

Cleaning column operator_owner


In [36]:
new_satellite_data['operator_owner'].unique()

array(['Hera Systems', 'AAC Clyde Space', 'Aalto University',
       'University of Aalborg', 'Asia Broadcast Satellite Ltd.',
       'Asher Space Research Institute at Technion/Israeli Aircraft Industries',
       'Austrian Space Forum (OEWF)/Spire',
       'National Reconnaissance Office (NRO)', 'US Air Force',
       'US Air Force ', 'Department of Homeland Security',
       'European Space Agency (ESA)', 'Aerospace Corporation',
       'EUTELSAT S.A.', 'Aurora Insight',
       'Center for Atmospheric Sciences, Hampton University/NASA',
       'Norwegian Coastal Admnistration', 'AISTech',
       'Al Yah Satellite Communications Co. (YAHSAT)',
       'Al-Farabi Kazakh National University',
       'Algerian Space Agency (ASAL)', 'Astro Live Experiences',
       'University of Brasilia', 'US Air Force Institute of Technology',
       'Algerian Space Agency (ASAL)/UK Space Agency',
       'Hispamar (subsidiary of Hispasat - Spain)',
       'National Institute for Space Research (INPE)',

In [37]:
new_satellite_data['operator_owner'] = new_satellite_data['operator_owner'].str.strip().str.lower()

Here we are having clean column data for Top 15 Operators, which we can use in Analysis

In [38]:
new_satellite_data['operator_owner'].value_counts().sort_values(ascending=False).head(15)

operator_owner
spacex                                       3989
oneweb satellites                             589
planet labs, inc.                             220
chinese ministry of national defense          149
spire global inc.                             135
ministry of defense                           117
swarm technologies                             90
iridium communications, inc.                   75
chang guang satellite technology co. ltd.      60
china academy of space technology (cast)       49
national reconnaissance office (nro)           48
indian space research organization (isro)      47
ses s.a.                                       46
european space agency (esa)                    40
satellogic s.a.                                38
Name: count, dtype: int64

Cleaning Column class_of_orbit

In [39]:
new_satellite_data['class_of_orbit'].unique()

array(['LEO', 'GEO', 'Elliptical', 'MEO', 'LEo'], dtype=object)

In [40]:
# LEO and LEo are same
new_satellite_data['class_of_orbit'] = new_satellite_data['class_of_orbit'].str.strip().str.upper()
new_satellite_data['class_of_orbit'].unique()

array(['LEO', 'GEO', 'ELLIPTICAL', 'MEO'], dtype=object)

Cleaning column type_of_orbit

In [41]:
new_satellite_data['type_of_orbit'].unique()

array(['Non-Polar Inclined', 'Sun-Synchronous', 'Unknown', 'Equatorial',
       'Polar', 'Molniya', 'Elliptical', 'Deep Highly Eccentric',
       'Retrograde', 'Cislunar', 'Sun-Synchronous near polar'],
      dtype=object)

In [42]:
# removing whitespaces and capitalizig first letter of word
new_satellite_data['type_of_orbit'] = new_satellite_data['type_of_orbit'].str.strip()
new_satellite_data['type_of_orbit'] = new_satellite_data['type_of_orbit'].str.title()
new_satellite_data['type_of_orbit'].unique()

array(['Non-Polar Inclined', 'Sun-Synchronous', 'Unknown', 'Equatorial',
       'Polar', 'Molniya', 'Elliptical', 'Deep Highly Eccentric',
       'Retrograde', 'Cislunar', 'Sun-Synchronous Near Polar'],
      dtype=object)

Cleaning column launch_vehicle

In [43]:
new_satellite_data['launch_vehicle'].unique()

array(['PSLV', 'Falcon 9', 'Soyuz-2.1a', 'Ariane 5 ECA', 'Atlas 3',
       'Proton', 'Delta 4 Heavy', 'Titan IVA', 'Titan IV', 'Atlas 5',
       'Vega', 'Electron', 'Nanorack Deployer', 'Dnepr', 'Ariane 5',
       'Vega W18', 'Pegasus XL', 'Soyuz-2.1b', 'Long March 3B',
       'Atlas 2AS', 'Proton M', 'Proton K', 'Zenit 3SLB', 'Zenit 2SB',
       'Delta 2310', 'Soyuz', 'Ariane 44L', 'Falcon Heavy',
       'Falcon 9/Sherpa FX', 'H2A', 'Breeze M', 'Ariane 5G', 'Epsilon',
       'PSLV C-30', 'PSLV-C29', 'Ceres 1', 'Long March 3C',
       'Long March 3A', 'Long March 2D', 'Kosmos 3M', 'PSLV C3',
       'PSLV XL', 'Soyuz-Fregat', 'Ariane 44LP', 'LauncherOne',
       'Long March 2B', 'Delta 2', 'Long March 11', 'PSLV C9', 'PSLV-CA',
       'PSLV C6', 'PSLV C7', 'Long March 4B', 'Kuaizhou', 'Long March 2C',
       'Space Shuttle (STS 93)', 'Long March 8', 'Kuaizhou-1A',
       'SEOPS Slingshot Deployer', 'PSLV-XL', 'Titan 2', 'Soyuz-2',
       'Minotaur-1', 'Delta 7000', 'Delta 7420', 'Rokot'

In [44]:
# strip spaces
new_satellite_data['launch_vehicle'] = new_satellite_data['launch_vehicle'].str.strip()

new_satellite_data.loc[new_satellite_data['launch_vehicle'].str.contains('Soyuz', na=False), 'launch_vehicle'] = 'Soyuz Family'
new_satellite_data.loc[new_satellite_data['launch_vehicle'].str.contains('PSLV', na=False), 'launch_vehicle'] = 'ISRO PSLV'
new_satellite_data.loc[new_satellite_data['launch_vehicle'].str.contains('Falcon', na=False), 'launch_vehicle'] = 'SpaceX Falcon'


new_satellite_data.loc[new_satellite_data['launch_vehicle'].str.contains('Long March', na=False), 'launch_vehicle'] = 'Long March Family'

new_satellite_data.loc[new_satellite_data['launch_vehicle'].str.contains('Ariane', na=False), 'launch_vehicle'] = 'Ariane Family'

new_satellite_data.loc[new_satellite_data['launch_vehicle'].str.contains('Atlas', na=False), 'launch_vehicle'] = 'Atlas Family'

new_satellite_data.loc[new_satellite_data['launch_vehicle'].str.contains('Delta', na=False), 'launch_vehicle'] = 'Delta Family'

new_satellite_data.loc[new_satellite_data['launch_vehicle'].str.contains('Proton', na=False), 'launch_vehicle'] = 'Proton Family'

new_satellite_data.loc[new_satellite_data['launch_vehicle'].str.contains('Zenit', na=False), 'launch_vehicle'] = 'Zenit Family'

new_satellite_data.loc[new_satellite_data['launch_vehicle'].str.contains('Minotaur', na=False), 'launch_vehicle'] = 'Minotaur Family'

new_satellite_data.loc[new_satellite_data['launch_vehicle'].str.contains('Kuaizhou', na=False), 'launch_vehicle'] = 'Kuaizhou Family'

new_satellite_data.loc[new_satellite_data['launch_vehicle'].str.contains('Ceres', na=False), 'launch_vehicle'] = 'Ceres Family'

new_satellite_data.loc[new_satellite_data['launch_vehicle'].str.contains('Vega', na=False), 'launch_vehicle'] = 'Vega Family'

new_satellite_data.loc[new_satellite_data['launch_vehicle'].str.contains('Titan', na=False), 'launch_vehicle'] = 'Titan Family'

new_satellite_data.loc[new_satellite_data['launch_vehicle'].str.contains('Space Shuttle', na=False), 'launch_vehicle'] = 'Space Shuttle Family'

new_satellite_data.loc[new_satellite_data['launch_vehicle'].str.contains('Deployer|Dispenser|L1011|Cygnus', na=False),
       'launch_vehicle'] = 'Non-Launch Systems'

new_satellite_data.loc[new_satellite_data['launch_vehicle'].str.contains('Breeze', na=False), 'launch_vehicle'] = 'Proton Family'

new_satellite_data['launch_vehicle'].unique()


array(['ISRO PSLV', 'SpaceX Falcon', 'Soyuz Family', 'Ariane Family',
       'Atlas Family', 'Proton Family', 'Delta Family', 'Titan Family',
       'Vega Family', 'Electron', 'Non-Launch Systems', 'Dnepr',
       'Pegasus XL', 'Long March Family', 'Zenit Family', 'H2A',
       'Epsilon', 'Ceres Family', 'Kosmos 3M', 'LauncherOne',
       'Kuaizhou Family', 'Space Shuttle Family', 'Minotaur Family',
       'Rokot', 'Pegasus', 'SSLV', 'Start 1', 'GSLV', 'GSLV MK.3',
       'GSLV Mk.2', 'JAXA M-V', 'Antares 230', 'Qased', 'Shavit',
       'Shavit 2', 'LVM3', 'Taurus', 'SEOPS Slingshot deployer',
       'Rocket 3.3', 'Nuri', 'Jielong 1', 'Fa', 'Tsyklon 3', 'Naro-1',
       'KT-2'], dtype=object)

In [45]:
new_satellite_data['launch_vehicle'].value_counts().head(15)

launch_vehicle
SpaceX Falcon         4757
Soyuz Family           709
Long March Family      616
ISRO PSLV              249
Ariane Family          193
Atlas Family           142
Proton Family          132
Electron               103
Delta Family            89
Vega Family             65
Dnepr                   61
H2A                     48
Rokot                   46
Non-Launch Systems      44
Zenit Family            35
Name: count, dtype: int64

Extract year from full date

In [ ]:
new_satellite_data['date_of_launch'] = pd.to_datetime(new_satellite_data['date_of_launch'], format='%m/%d/%y')
new_satellite_data['launch_year'] = new_satellite_data['date_of_launch'].dt.year
new_satellite_data['launch_year'] = (new_satellite_data['date_of_launch'].dt.year.astype('Int64'))

In [49]:
new_satellite_data.head(5)

,current_official_name_of_satellite,country_of_operator_owner,operator_owner,users,purpose,class_of_orbit,type_of_orbit,perigee_km,apogee_km,perigee_km,launch_mass_kg,date_of_launch,expected_lifetime_yrs,launch_vehicle,norad_number,launch_year
0,1HOPSAT-TD,USA,hera systems,Commercial,Earth Observation,LEO,Non-Polar Inclined,566.000,576.000,566.000,22.00,2019-12-11,4.0,ISRO PSLV,44859,2019
1,AAC AIS-Sat1 (Kelpie 1),United Kingdom,aac clyde space,Commercial,Earth Observation,LEO,Sun-Synchronous,637.000,654.000,637.000,4.00,2023-01-03,4.0,SpaceX Falcon,55107,2023
2,Aalto-1,Finland,aalto university,Civil,Technology Development,LEO,Sun-Synchronous,497.000,517.000,497.000,5.00,2017-06-23,2.0,ISRO PSLV,42775,2017
3,AAt-4,Denmark,university of aalborg,Civil,Earth Observation,LEO,Sun-Synchronous,442.000,687.000,442.000,1.00,2016-04-25,4.0,Soyuz Family,41460,2016
4,ABS-2,Multinational,asia broadcast satellite ltd.,Commercial,Communications,GEO,Unknown,35.778,35.793,35.778,6.33,2014-02-06,15.0,Ariane Family,39508,2014


# **Step 2.2: Cleaning of Live (KPI) Dataset from Celestrak: NORAD**

In [57]:
df_live.head(100)

,OBJECT_NAME,OBJECT_ID,EPOCH,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,EPHEMERIS_TYPE,CLASSIFICATION_TYPE,NORAD_CAT_ID,ELEMENT_SET_NO,REV_AT_EPOCH,BSTAR,MEAN_MOTION_DOT,MEAN_MOTION_DDOT
0,STARLINK-1008,2019-074B,2026-04-16T03:10:57.891072,15.348440,0.000317,53.1552,4.1211,159.2750,200.8382,0,U,44714,999,35461,0.000787,0.000267,0.000000
1,STARLINK-1012,2019-074F,2026-04-16T03:10:12.397152,15.341506,0.000368,53.1602,4.4007,144.0726,216.0525,0,U,44718,999,35460,0.000453,0.000148,0.000000
2,STARLINK-1017,2019-074L,2026-04-16T01:46:57.236448,15.275657,0.000332,53.0511,358.5182,153.6459,206.4708,0,U,44723,999,35467,0.000817,0.000220,0.000000
3,STARLINK-1019,2019-074M,2026-04-16T00:00:02.000160,15.956265,0.000498,53.0355,307.2820,279.7659,72.5714,0,U,44724,999,603,0.000607,0.002470,0.000035
4,STARLINK-1020,2019-074N,2026-04-16T04:41:50.795520,15.332561,0.000293,53.1607,29.3252,146.0752,214.0438,0,U,44725,999,35424,0.000875,0.000282,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,STARLINK-1368,2020-025K,2026-04-15T21:07:23.867904,15.328250,0.000083,53.1585,288.4219,59.8496,300.2589,0,U,45540,999,33086,0.000993,0.000317,0.000000
96,STARLINK-1371,2020-025M,2026-04-15T18:04:03.184032,15.534735,0.000248,53.0465,271.5088,280.6153,79.4577,0,U,45542,999,33108,0.000955,0.000616,0.000000
97,STARLINK-1372,2020-025N,2026-04-15T22:05:19.824576,15.332254,0.000056,53.1589,287.9118,68.6467,291.4595,0,U,45543,999,32997,0.001000,0.000323,0.000000
98,STARLINK-1373,2020-025P,2026-04-15T17:41:26.347200,15.576844,0.000191,53.0458,270.0749,222.0693,138.0172,0,U,45544,999,33110,0.000905,0.000683,0.000000


In [58]:
df_live.tail(100)

,OBJECT_NAME,OBJECT_ID,EPOCH,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,EPHEMERIS_TYPE,CLASSIFICATION_TYPE,NORAD_CAT_ID,ELEMENT_SET_NO,REV_AT_EPOCH,BSTAR,MEAN_MOTION_DOT,MEAN_MOTION_DDOT
10048,STARLINK-37180,2026-059N,2026-04-16T06:00:02.000160,15.493120,0.000119,53.1600,175.6715,61.1421,182.5686,0,U,68333,999,508,0.000332,0.000180,0.0
10049,STARLINK-37191,2026-059P,2026-04-16T06:00:02.000160,15.599794,0.000128,53.1600,174.9169,96.8400,158.2612,0,U,68334,999,508,-0.025895,-0.018853,0.0
10050,STARLINK-37217,2026-059Q,2026-04-16T06:00:02.000160,15.590626,0.000144,53.1600,174.9425,95.9227,146.5402,0,U,68335,999,508,-0.027004,-0.018965,0.0
10051,STARLINK-37143,2026-059R,2026-04-16T06:00:02.000160,15.631021,0.000103,53.1601,174.7952,81.5813,233.7745,0,U,68336,999,508,-0.023316,-0.019140,0.0
10052,STARLINK-36237,2026-059S,2026-04-16T06:00:02.000160,15.677124,0.000083,53.1599,174.6614,81.8902,299.1667,0,U,68337,999,508,-0.019350,-0.019135,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10143,STARLINK-37212,2026-070AA,2026-04-16T06:00:02.000160,15.855502,0.000091,53.1585,195.9614,88.5199,14.1448,0,U,68566,999,341,-0.006400,-0.013784,0.0
10144,STARLINK-37261,2026-070AB,2026-04-16T06:00:02.000160,15.855520,0.000093,53.1587,195.9526,88.4725,18.5119,0,U,68567,999,341,-0.006419,-0.013822,0.0
10145,STARLINK-37283,2026-070AC,2026-04-16T06:00:02.000160,15.809185,0.000117,53.1595,196.3456,100.9612,176.2154,0,U,68568,999,342,-0.000315,-0.000626,0.0
10146,STARLINK-37256,2026-070AD,2026-04-16T06:00:02.000160,15.809068,0.000141,53.1596,196.3540,92.0571,180.8487,0,U,68569,999,342,-0.000205,-0.000411,0.0


In [59]:
df_live.columns

Index(['OBJECT_NAME', 'OBJECT_ID', 'EPOCH', 'MEAN_MOTION', 'ECCENTRICITY',
       'INCLINATION', 'RA_OF_ASC_NODE', 'ARG_OF_PERICENTER', 'MEAN_ANOMALY',
       'EPHEMERIS_TYPE', 'CLASSIFICATION_TYPE', 'NORAD_CAT_ID',
       'ELEMENT_SET_NO', 'REV_AT_EPOCH', 'BSTAR', 'MEAN_MOTION_DOT',
       'MEAN_MOTION_DDOT'],
      dtype='object')

#remove unnecessary columns

In [60]:
df_live = df_live.loc[:, ~df_live.columns.str.contains("EPHEMERIS_TYPE")]
df_live.columns

Index(['OBJECT_NAME', 'OBJECT_ID', 'EPOCH', 'MEAN_MOTION', 'ECCENTRICITY',
       'INCLINATION', 'RA_OF_ASC_NODE', 'ARG_OF_PERICENTER', 'MEAN_ANOMALY',
       'CLASSIFICATION_TYPE', 'NORAD_CAT_ID', 'ELEMENT_SET_NO', 'REV_AT_EPOCH',
       'BSTAR', 'MEAN_MOTION_DOT', 'MEAN_MOTION_DDOT'],
      dtype='object')

In [61]:
# List of columns to remove (handling the naming exactly as they appear in your data)
cols_to_remove = ['EPOCH', 'RA_OF_ASC_NODE', 'CLASSIFICATION_TYPE', 'ELEMENT_SET_NO','REV_AT_EPOCH','MEAN_MOTION_DDOT','MEAN_MOTION_DOT','BSTAR','MEAN_ANOMALY']

# Drop the columns
df_live = df_live.drop(columns=cols_to_remove, errors='ignore')
df_live.columns

Index(['OBJECT_NAME', 'OBJECT_ID', 'MEAN_MOTION', 'ECCENTRICITY',
       'INCLINATION', 'ARG_OF_PERICENTER', 'NORAD_CAT_ID'],
      dtype='object')

In [62]:
df_live

,OBJECT_NAME,OBJECT_ID,MEAN_MOTION,ECCENTRICITY,INCLINATION,ARG_OF_PERICENTER,NORAD_CAT_ID
0,STARLINK-1008,2019-074B,15.348440,0.000317,53.1552,159.2750,44714
1,STARLINK-1012,2019-074F,15.341506,0.000368,53.1602,144.0726,44718
2,STARLINK-1017,2019-074L,15.275657,0.000332,53.0511,153.6459,44723
3,STARLINK-1019,2019-074M,15.956265,0.000498,53.0355,279.7659,44724
4,STARLINK-1020,2019-074N,15.332561,0.000293,53.1607,146.0752,44725
...,...,...,...,...,...,...,...
10143,STARLINK-37212,2026-070AA,15.855502,0.000091,53.1585,88.5199,68566
10144,STARLINK-37261,2026-070AB,15.855520,0.000093,53.1587,88.4725,68567
10145,STARLINK-37283,2026-070AC,15.809185,0.000117,53.1595,100.9612,68568
10146,STARLINK-37256,2026-070AD,15.809068,0.000141,53.1596,92.0571,68569


standardize collumns

In [63]:
df_live.columns = df_live.columns.str.lower()
df_live

,object_name,object_id,mean_motion,eccentricity,inclination,arg_of_pericenter,norad_cat_id
0,STARLINK-1008,2019-074B,15.348440,0.000317,53.1552,159.2750,44714
1,STARLINK-1012,2019-074F,15.341506,0.000368,53.1602,144.0726,44718
2,STARLINK-1017,2019-074L,15.275657,0.000332,53.0511,153.6459,44723
3,STARLINK-1019,2019-074M,15.956265,0.000498,53.0355,279.7659,44724
4,STARLINK-1020,2019-074N,15.332561,0.000293,53.1607,146.0752,44725
...,...,...,...,...,...,...,...
10143,STARLINK-37212,2026-070AA,15.855502,0.000091,53.1585,88.5199,68566
10144,STARLINK-37261,2026-070AB,15.855520,0.000093,53.1587,88.4725,68567
10145,STARLINK-37283,2026-070AC,15.809185,0.000117,53.1595,100.9612,68568
10146,STARLINK-37256,2026-070AD,15.809068,0.000141,53.1596,92.0571,68569


check for duplicates

In [64]:
df_live.duplicated().sum()

np.int64(0)

checking for null values

In [65]:
round((df_live.isna().sum() / len(df_live)) * 100)

object_name          0.0
object_id            0.0
mean_motion          0.0
eccentricity         0.0
inclination          0.0
arg_of_pericenter    0.0
norad_cat_id         0.0
dtype: float64

In [66]:
df_live.describe()

,mean_motion,eccentricity,inclination,arg_of_pericenter,norad_cat_id
count,10148.000000,10148.000000,10148.000000,10148.000000,10148.000000
mean,15.281047,0.000163,54.587406,168.008543,59661.726646
std,0.194522,0.000134,15.074469,92.731929,6071.632053
min,14.950825,0.000003,42.983900,0.376300,44714.000000
25%,15.088871,0.000111,43.002900,85.973575,55450.750000
50%,15.275878,0.000132,53.159100,104.686450,60028.500000
75%,15.302035,0.000168,53.216700,268.829750,64803.250000
max,16.325271,0.003266,97.662400,359.787700,68570.000000


In [67]:
df_live["object_name_unique"] = df_live["object_name"].str.replace(r"[^a-zA-Z]", "", regex=True)
df_live

,object_name,object_id,mean_motion,eccentricity,inclination,arg_of_pericenter,norad_cat_id,object_name_unique
0,STARLINK-1008,2019-074B,15.348440,0.000317,53.1552,159.2750,44714,STARLINK
1,STARLINK-1012,2019-074F,15.341506,0.000368,53.1602,144.0726,44718,STARLINK
2,STARLINK-1017,2019-074L,15.275657,0.000332,53.0511,153.6459,44723,STARLINK
3,STARLINK-1019,2019-074M,15.956265,0.000498,53.0355,279.7659,44724,STARLINK
4,STARLINK-1020,2019-074N,15.332561,0.000293,53.1607,146.0752,44725,STARLINK
...,...,...,...,...,...,...,...,...
10143,STARLINK-37212,2026-070AA,15.855502,0.000091,53.1585,88.5199,68566,STARLINK
10144,STARLINK-37261,2026-070AB,15.855520,0.000093,53.1587,88.4725,68567,STARLINK
10145,STARLINK-37283,2026-070AC,15.809185,0.000117,53.1595,100.9612,68568,STARLINK
10146,STARLINK-37256,2026-070AD,15.809068,0.000141,53.1596,92.0571,68569,STARLINK


determin if "Starlink" is an exclusive value or there are other names in the API

In [68]:
df_live["object_name_unique"].unique()

array(['STARLINK', 'STARLINKDTC'], dtype=object)

Test to see how many columns were joined

In [69]:
# Verify columns and dtypes first
print("df_live columns:", df_live.columns.tolist())
print("df_live['norad_cat_id'] dtype:", df_live['norad_cat_id'].dtype)
print("new_satellite_data columns:", new_satellite_data.columns.tolist())
print("new_satellite_data['norad_number'] dtype:", new_satellite_data['norad_number'].dtype)

# Ensure join columns are string/object type
df_live['norad_cat_id'] = df_live['norad_cat_id'].astype(str)
new_satellite_data['norad_number'] = new_satellite_data['norad_number'].astype(str)

# Outer merge with indicator
merged = new_satellite_data.merge(
    df_live,
    left_on="norad_number",
    right_on="norad_cat_id",
    how="inner",
    indicator=True  # Tracks join status
)

# Join stats
print("\nJoin counts:")
print(merged['_merge'].value_counts())
print("unique values", merged["_merge"].unique())

merged.tail(100)



df_live columns: ['object_name', 'object_id', 'mean_motion', 'eccentricity', 'inclination', 'arg_of_pericenter', 'norad_cat_id', 'object_name_unique']
df_live['norad_cat_id'] dtype: int64
new_satellite_data columns: ['current_official_name_of_satellite', 'country_of_operator_owner', 'operator_owner', 'users', 'purpose', 'class_of_orbit', 'type_of_orbit', 'perigee_km', 'apogee_km', 'perigee_km', 'launch_mass_kg', 'date_of_launch', 'expected_lifetime_yrs', 'launch_vehicle', 'norad_number', 'launch_year']
new_satellite_data['norad_number'] dtype: int64

Join counts:
_merge
both          2931
left_only        0
right_only       0
Name: count, dtype: int64
unique values ['both']
Categories (3, object): ['left_only', 'right_only', 'both']


,current_official_name_of_satellite,country_of_operator_owner,operator_owner,users,purpose,class_of_orbit,type_of_orbit,perigee_km,apogee_km,perigee_km,...,launch_year,object_name,object_id,mean_motion,eccentricity,inclination,arg_of_pericenter,norad_cat_id,object_name_unique,_merge
2831,Starlink-5902,USA,spacex,Commercial,Communications,LEO,Non-Polar Inclined,565.0,567.0,565.0,...,2023,STARLINK-5902,2023-042N,15.267425,0.000161,43.0018,272.7905,55998,STARLINK,both
2832,Starlink-5903,USA,spacex,Commercial,Communications,LEO,Non-Polar Inclined,565.0,567.0,565.0,...,2023,STARLINK-5903,2023-042L,15.266641,0.000201,43.0030,276.8320,55996,STARLINK,both
2833,Starlink-5905,USA,spacex,Commercial,Communications,LEO,Non-Polar Inclined,565.0,567.0,565.0,...,2023,STARLINK-5905,2023-042A,15.275539,0.000140,43.0019,278.2798,55986,STARLINK,both
2834,Starlink-5906,USA,spacex,Commercial,Communications,LEO,Polar,577.0,581.0,577.0,...,2023,STARLINK-5906,2023-037BA,14.983208,0.000316,69.9998,282.5351,55962,STARLINK,both
2835,Starlink-5907,USA,spacex,Commercial,Communications,LEO,Non-Polar Inclined,565.0,567.0,565.0,...,2023,STARLINK-5907,2023-042AM,15.275854,0.000190,43.0018,296.7051,56021,STARLINK,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2926,Starlink-6120,USA,spacex,Commercial,Communications,LEO,Non-Polar Inclined,564.0,567.0,564.0,...,2023,STARLINK-6120,2023-046AR,15.276738,0.000241,43.0065,255.2812,56131,STARLINK,both
2927,Starlink-6121,USA,spacex,Commercial,Communications,LEO,Non-Polar Inclined,564.0,567.0,564.0,...,2023,STARLINK-6121,2023-046AP,15.275637,0.000133,43.0015,268.9970,56129,STARLINK,both
2928,Starlink-6122,USA,spacex,Commercial,Communications,LEO,Non-Polar Inclined,564.0,567.0,564.0,...,2023,STARLINK-6122,2023-046AN,15.267395,0.000157,43.0020,264.5657,56128,STARLINK,both
2929,Starlink-6124,USA,spacex,Commercial,Communications,LEO,Non-Polar Inclined,564.0,567.0,564.0,...,2023,STARLINK-6124,2023-046AL,15.267423,0.000096,43.0021,269.7257,56126,STARLINK,both


In [70]:
df_live.columns

Index(['object_name', 'object_id', 'mean_motion', 'eccentricity',
       'inclination', 'arg_of_pericenter', 'norad_cat_id',
       'object_name_unique'],
      dtype='object')

In [71]:
new_satellite_data.columns

Index(['current_official_name_of_satellite', 'country_of_operator_owner',
       'operator_owner', 'users', 'purpose', 'class_of_orbit', 'type_of_orbit',
       'perigee_km', 'apogee_km', 'perigee_km', 'launch_mass_kg',
       'date_of_launch', 'expected_lifetime_yrs', 'launch_vehicle',
       'norad_number', 'launch_year'],
      dtype='object')

In [72]:
merged.columns

Index(['current_official_name_of_satellite', 'country_of_operator_owner',
       'operator_owner', 'users', 'purpose', 'class_of_orbit', 'type_of_orbit',
       'perigee_km', 'apogee_km', 'perigee_km', 'launch_mass_kg',
       'date_of_launch', 'expected_lifetime_yrs', 'launch_vehicle',
       'norad_number', 'launch_year', 'object_name', 'object_id',
       'mean_motion', 'eccentricity', 'inclination', 'arg_of_pericenter',
       'norad_cat_id', 'object_name_unique', '_merge'],
      dtype='object')

extract the years and add them to a new collumn

In [73]:
merged

,current_official_name_of_satellite,country_of_operator_owner,operator_owner,users,purpose,class_of_orbit,type_of_orbit,perigee_km,apogee_km,perigee_km,...,launch_year,object_name,object_id,mean_motion,eccentricity,inclination,arg_of_pericenter,norad_cat_id,object_name_unique,_merge
0,Starlink-1008,USA,spacex,Commercial,Communications,LEO,Non-Polar Inclined,549.0,551.0,549.0,...,2019,STARLINK-1008,2019-074B,15.348440,0.000317,53.1552,159.2750,44714,STARLINK,both
1,Starlink-1012,USA,spacex,Commercial,Communications,LEO,Non-Polar Inclined,549.0,551.0,549.0,...,2019,STARLINK-1012,2019-074F,15.341506,0.000368,53.1602,144.0726,44718,STARLINK,both
2,Starlink-1017,USA,spacex,Commercial,Communications,LEO,Non-Polar Inclined,549.0,551.0,549.0,...,2019,STARLINK-1017,2019-074L,15.275657,0.000332,53.0511,153.6459,44723,STARLINK,both
3,Starlink-1019,USA,spacex,Commercial,Communications,LEO,Non-Polar Inclined,549.0,551.0,549.0,...,2019,STARLINK-1019,2019-074M,15.956265,0.000498,53.0355,279.7659,44724,STARLINK,both
4,Starlink-1020,USA,spacex,Commercial,Communications,LEO,Non-Polar Inclined,548.0,551.0,548.0,...,2019,STARLINK-1020,2019-074N,15.332561,0.000293,53.1607,146.0752,44725,STARLINK,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2926,Starlink-6120,USA,spacex,Commercial,Communications,LEO,Non-Polar Inclined,564.0,567.0,564.0,...,2023,STARLINK-6120,2023-046AR,15.276738,0.000241,43.0065,255.2812,56131,STARLINK,both
2927,Starlink-6121,USA,spacex,Commercial,Communications,LEO,Non-Polar Inclined,564.0,567.0,564.0,...,2023,STARLINK-6121,2023-046AP,15.275637,0.000133,43.0015,268.9970,56129,STARLINK,both
2928,Starlink-6122,USA,spacex,Commercial,Communications,LEO,Non-Polar Inclined,564.0,567.0,564.0,...,2023,STARLINK-6122,2023-046AN,15.267395,0.000157,43.0020,264.5657,56128,STARLINK,both
2929,Starlink-6124,USA,spacex,Commercial,Communications,LEO,Non-Polar Inclined,564.0,567.0,564.0,...,2023,STARLINK-6124,2023-046AL,15.267423,0.000096,43.0021,269.7257,56126,STARLINK,both


**Cleaning 3rd API Data**

In [74]:
space_track_data.shape
space_track_data.head(30)

,INTLDES,NORAD_CAT_ID,OBJECT_TYPE,SATNAME,COUNTRY,LAUNCH,SITE,DECAY,PERIOD,INCLINATION,...,RCSVALUE,RCS_SIZE,FILE,LAUNCH_YEAR,LAUNCH_NUM,LAUNCH_PIECE,CURRENT,OBJECT_NAME,OBJECT_ID,OBJECT_NUMBER
0,1957-001A,1,ROCKET BODY,SL-1 R/B,CIS,1957-10-04,TTMTR,1957-12-01,96.19,65.10,...,0,LARGE,1,1957,1,A,Y,SL-1 R/B,1957-001A,1
1,1957-001B,2,PAYLOAD,SPUTNIK 1,CIS,1957-10-04,TTMTR,1958-01-03,96.10,65.00,...,0,None,7179,1957,1,B,Y,SPUTNIK 1,1957-001B,2
2,1957-002A,3,PAYLOAD,SPUTNIK 2,CIS,1957-11-03,TTMTR,1958-04-14,103.74,65.33,...,0,SMALL,9221,1957,2,A,Y,SPUTNIK 2,1957-002A,3
3,1958-001A,4,PAYLOAD,EXPLORER 1,US,1958-02-01,AFETR,1970-03-31,88.48,33.15,...,0,None,1,1958,1,A,Y,EXPLORER 1,1958-001A,4
4,1958-002A,16,ROCKET BODY,VANGUARD R/B,US,1958-03-17,AFETR,None,137.20,34.26,...,0,MEDIUM,9430,1958,2,A,Y,VANGUARD R/B,1958-002A,16
5,1958-002B,5,PAYLOAD,VANGUARD 1,US,1958-03-17,AFETR,None,132.60,34.25,...,0,SMALL,9450,1958,2,B,Y,VANGUARD 1,1958-002B,5
6,1958-002C,1576,DEBRIS,VANGUARD DEB,US,1958-03-17,AFETR,None,121.76,34.22,...,0,SMALL,9446,1958,2,C,Y,VANGUARD DEB,1958-002C,1576
7,1958-003A,6,PAYLOAD,EXPLORER 3,US,1958-03-26,AFETR,1958-06-28,151.74,19.24,...,0,SMALL,9295,1958,3,A,Y,EXPLORER 3,1958-003A,6
8,1958-004A,7,ROCKET BODY,SL-1 R/B,CIS,1958-05-15,TTMTR,1958-12-03,102.74,65.14,...,0,None,1,1958,4,A,Y,SL-1 R/B,1958-004A,7
9,1958-004B,8,PAYLOAD,SPUTNIK 3,CIS,1958-05-15,TTMTR,1960-04-06,88.43,65.06,...,0,LARGE,1,1958,4,B,Y,SPUTNIK 3,1958-004B,8


In [75]:
space_track_data.columns

Index(['INTLDES', 'NORAD_CAT_ID', 'OBJECT_TYPE', 'SATNAME', 'COUNTRY',
       'LAUNCH', 'SITE', 'DECAY', 'PERIOD', 'INCLINATION', 'APOGEE', 'PERIGEE',
       'COMMENT', 'COMMENTCODE', 'RCSVALUE', 'RCS_SIZE', 'FILE', 'LAUNCH_YEAR',
       'LAUNCH_NUM', 'LAUNCH_PIECE', 'CURRENT', 'OBJECT_NAME', 'OBJECT_ID',
       'OBJECT_NUMBER'],
      dtype='object')

In [76]:
space_track_data.columns = (space_track_data.columns.str.lower()          # lowercase
                        .str.strip()          # remove leading/trailing spaces
                        .str.replace(" ", "_")  # replace spaces with underscore
)

Drop unwanted columns

In [87]:
cols_to_drop = [
    "file",
    "rcs_size",
    "rcsvalue",
    "comment",
    "commentcode",
    "launch_piece",
    "launch_num",
    "intldes",
    "launch",
    "satname",
    'site', 
    'decay',
    'period', 
    'inclination', 
    'apogee', 
    'perigee',
    'current',
    'object_name', 
    'object_id', 
    'object_number'
]

space_track_data = space_track_data.drop(columns=cols_to_drop, errors="ignore")

In [89]:
space_track_data.shape

(68568, 4)

In [90]:
space_track_data.duplicated().sum()

np.int64(0)

In [91]:
round((space_track_data.isna().sum() / len(space_track_data)) * 100)

norad_cat_id    0.0
object_type     0.0
country         0.0
launch_year     0.0
dtype: float64

In [92]:
space_track_data.head(30)

,norad_cat_id,object_type,country,launch_year
0,1,ROCKET BODY,CIS,1957
1,2,PAYLOAD,CIS,1957
2,3,PAYLOAD,CIS,1957
3,4,PAYLOAD,US,1958
4,16,ROCKET BODY,US,1958
5,5,PAYLOAD,US,1958
6,1576,DEBRIS,US,1958
7,6,PAYLOAD,US,1958
8,7,ROCKET BODY,CIS,1958
9,8,PAYLOAD,CIS,1958


In [94]:
# Verify columns and dtypes first
print("space_track_data columns:", space_track_data.columns.tolist())
print("space_track_data['norad_cat_id'] dtype:", space_track_data['norad_cat_id'].dtype)
print("new_satellite_data columns:", new_satellite_data.columns.tolist())
print("new_satellite_data['norad_number'] dtype:", new_satellite_data['norad_number'].dtype)

# Ensure join columns are string/object type
space_track_data['norad_cat_id'] = space_track_data['norad_cat_id'].astype(str)
new_satellite_data['norad_number'] = new_satellite_data['norad_number'].astype(str)

# Outer merge with indicator
second_merged = space_track_data.merge(
    new_satellite_data,
    left_on="norad_cat_id",
    right_on="norad_number",
    how="inner",
    indicator=True  # Tracks join status
)

# Join stats
print("\nJoin counts:")
print(second_merged['_merge'].value_counts())
print("unique values", second_merged["_merge"].unique())

second_merged.tail(100)

space_track_data columns: ['norad_cat_id', 'object_type', 'country', 'launch_year']
space_track_data['norad_cat_id'] dtype: object
new_satellite_data columns: ['current_official_name_of_satellite', 'country_of_operator_owner', 'operator_owner', 'users', 'purpose', 'class_of_orbit', 'type_of_orbit', 'perigee_km', 'apogee_km', 'perigee_km', 'launch_mass_kg', 'date_of_launch', 'expected_lifetime_yrs', 'launch_vehicle', 'norad_number', 'launch_year']
new_satellite_data['norad_number'] dtype: object

Join counts:
_merge
both          7553
left_only        0
right_only       0
Name: count, dtype: int64
unique values ['both']
Categories (3, object): ['left_only', 'right_only', 'both']


,norad_cat_id,object_type,country,launch_year_x,current_official_name_of_satellite,country_of_operator_owner,operator_owner,users,purpose,class_of_orbit,...,perigee_km,apogee_km,perigee_km,launch_mass_kg,date_of_launch,expected_lifetime_yrs,launch_vehicle,norad_number,launch_year_y,_merge
7453,56214,PAYLOAD,LTU,2023,LacunaSat-2F,Lithuania,lacuna space/nanoavionics,Commercial,Communications,LEO,...,487.0,506.0,487.0,4.0,2023-04-15,4.0,SpaceX Falcon,56214,2023,both
7454,56215,PAYLOAD,BRAZ,2023,VCUB-1,Brazil,visiona tecnologia espacial s.a.,Commercial,Technology Demonstration,LEO,...,490.0,505.0,490.0,10.0,2023-04-15,4.0,SpaceX Falcon,56215,2023,both
7455,56216,PAYLOAD,UK,2023,ELO-3,France,clyde space ltd./eutelsat,Commercial,Technology Demonstration,LEO,...,496.0,507.0,496.0,10.0,2023-04-15,4.0,SpaceX Falcon,56216,2023,both
7456,56217,PAYLOAD,CA,2023,Kepler-20,Canada,kepler communications,Commercial,Communications,LEO,...,491.0,507.0,491.0,15.0,2023-04-15,4.0,SpaceX Falcon,56217,2023,both
7457,56218,PAYLOAD,CA,2023,Kepler-21,Canada,kepler communications,Commercial,Communications,LEO,...,NaN,NaN,NaN,15.0,2023-04-15,4.0,SpaceX Falcon,56218,2023,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7548,56338,PAYLOAD,US,2023,Starlink-5554,USA,spacex,Commercial,Communications,LEO,...,569.0,571.0,569.0,300.0,2023-04-27,4.0,SpaceX Falcon,56338,2023,both
7549,56339,PAYLOAD,US,2023,Starlink-5552,USA,spacex,Commercial,Communications,LEO,...,569.0,571.0,569.0,300.0,2023-04-27,4.0,SpaceX Falcon,56339,2023,both
7550,56340,PAYLOAD,US,2023,Starlink-5540,USA,spacex,Commercial,Communications,LEO,...,569.0,571.0,569.0,300.0,2023-04-27,4.0,SpaceX Falcon,56340,2023,both
7551,62637,PAYLOAD,US,2025,Starlink-3961,USA,spacex,Commercial,Communications,LEO,...,538.0,541.0,538.0,260.0,2022-05-14,4.0,SpaceX Falcon,62637,2022,both


Saving Cleaned file for further analysis

In [97]:
df_live.to_csv("Output_Files/df_live.csv", index=False)
space_track_data.to_csv("Output_Files/space_track_data.csv", index=False)
new_satellite_data.to_csv("Output_Files/cleaned_satellite_data.csv", index=False)
